In [3]:
# 4 bus network simulation

import numpy as np

In [4]:
buses = ["A", "B", "C", "D"]

# Each tuple defines the positive flow direction
lines = [
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "D"),
    ("B", "D")
]

x = np.ones(5)  # Reactance of each line


In [5]:
# incidence matrix

K = np.zeros((5, 4))

for i, (start, end) in enumerate(lines):
    K[i, buses.index(start)] = 1
    K[i, buses.index(end)] = -1

print(K)


[[ 1. -1.  0.  0.]
 [ 0.  1. -1.  0.]
 [ 0.  0.  1. -1.]
 [ 1.  0.  0. -1.]
 [ 0.  1.  0. -1.]]


In [6]:
# bus susceptence matrix

W = np.diag(1 / x)

B = K.T @ W @ K

print(B)


[[ 2. -1.  0. -1.]
 [-1.  3. -1. -1.]
 [ 0. -1.  2. -1.]
 [-1. -1. -1.  3.]]


In [7]:
B_reduced = B[:3, :3]

p = np.array([1, 0, 0])

theta_reduced = np.linalg.solve(B_reduced, p)

theta = np.append(theta_reduced, 0)

f = np.diag(1 / x) @ K @ theta

f
#print(theta)
#np.linalg.solve(B_)

array([0.375, 0.125, 0.125, 0.625, 0.25 ])

In [9]:
thetas = np.linalg.solve(B_reduced, np.eye(3))

thetas_full = np.append(thetas, [np.zeros(3)], axis = 0)

ptdf = np.diag(1 / x) @ K @ thetas_full
#last_row = np.zeros(3)
#np.vstack(thetas, last_row)

In [11]:
f = W @ K @ theta

for line, flow in zip(lines, f):
    print(f"{line}: {flow:.3f} MW")


('A', 'B'): 0.375 MW
('B', 'C'): 0.125 MW
('C', 'D'): 0.125 MW
('A', 'D'): 0.625 MW
('B', 'D'): 0.250 MW


In [49]:
lines = [
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "D")
    #("B", "D")
]

K_new = np.zeros((len(lines), len(buses)))

for i, (start, end) in enumerate(lines):
    K_new[i, buses.index(start)] = 1
    K_new[i, buses.index(end)] = -1
    #print(f"{i } {start } {end }")

B_new = K_new.T @ K_new

B_newr = B_new[:3, :3]

thetas = np.linalg.solve(B_newr, np.eye(3))

thetas_full = np.vstack( (thetas, np.zeros(3)) )

K_new @ thetas_full

array([[ 0.25, -0.5 , -0.25],
       [ 0.25,  0.5 , -0.25],
       [ 0.25,  0.5 ,  0.75],
       [ 0.75,  0.5 ,  0.25]])

In [65]:
#lodf = ptdf[:4,1]/(1-.5)

#print(f"col 1: {ptdf[:4,0]/(1-ptdf[4,0])}")
#print(f"col 2: {ptdf[:4,1]/(1-ptdf[4,1])}")
#ptdf[:4,2]/(1-ptdf[4,2])
#f[:4] + lodf * .25

lodf = ptdf[:4,1]/(1 - ptdf[4,1])

po_ptdf = ptdf[:4, :] + np.outer(lodf, ptdf[4, :]) # post outage ptdf


In [70]:
from scipy.optimize import linprog

In [ ]:
# Generator costs at A and C
c = [25, 85]

# Extract the A-D line sensitivities
# Row 3 = A-D line
# Columns 0 and 2 = generators at A and C
base_AD = ptdf[3, [0, 2]]
outage_AD = po_ptdf[3, [0, 2]]

# Transmission constraints: A_ub @ g <= b_ub
A_ub = np.vstack([base_AD, outage_AD])
b_ub = [10.5, 10.5]

# Energy balance: g_A + g_C = 16
A_eq = [[1, 1]]
b_eq = [16]
bounds = [(0, 16), (0, 16)]

result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)



NameError: name 'ptdf' is not defined

In [87]:
print(result.ineqlin.residual)
print(result.ineqlin.marginals)
print(result.eqlin.marginals)

[2. 0.]
[  -0. -120.]
[115.]
